# 00 · Premissas

**Objetivo:** Revisar e documentar todas as premissas do modelo antes de qualquer cálculo.

**Entradas:** `config/premissas.yaml`

**Saídas:** Premissas validadas, com fonte de cada valor (ou marcada como pendente).

## Passos

- [x] Validar o arquivo (tipos, faixas de sanidade, coerência entre campos)
- [x] Listar as premissas ainda sem fonte (`FONTE PENDENTE` no YAML)
- [x] Conferir a composição do lance e o complemento no cartão (bate com a planilha)
- [ ] Conferir os dados da cota de consórcio no extrato da administradora
- [ ] Confirmar `fator_total_devedor` (total devedor ÷ carta)
- [ ] Revisar km/ano e taxa de desconto real
- [ ] Registrar a fonte de cada valor alterado

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # permite importar src/ a partir de notebooks/

from src.config import carregar_premissas, fontes_pendentes

# carregar_premissas já valida o arquivo: se algo estiver errado,
# para aqui com a lista completa de problemas.
p = carregar_premissas()
print("Premissas carregadas e validadas.")

## Premissas atuais

In [ ]:
import json
print(json.dumps(p, indent=2, ensure_ascii=False))

## Premissas sem fonte

Cada linha abaixo está marcada com `FONTE PENDENTE` no YAML. Ao confirmar a fonte,
troque o marcador pela referência (documento, link, data).

In [ ]:
pendentes = fontes_pendentes()
print(f"{len(pendentes)} premissa(s) sem fonte:")
for linha in pendentes:
    print("-", linha)

## Conferência: composição do lance e complemento no cartão

Como a carta é usada (lógica da aba `simulação_aumento_credito`):

- O **lance total** (54% da carta) tem duas partes: o **lance embutido** (25%), que sai da própria
  carta, e o restante (29%), pago com **recurso próprio**.
- O **crédito líquido** é a carta menos o lance embutido.
- O que sobra do recurso próprio depois do lance ajuda a pagar o carro.
- Se o carro custa mais que crédito líquido + sobra, a diferença vai para o **cartão**, em 12×,
  com 20% de acréscimo sobre o total.
- Se o carro custa menos, não há cartão: a sobra **abate o saldo devedor** do consórcio.

Mostramos o complemento nas duas pontas da faixa do universo. Os cálculos ficam em `src/consorcio.py`,
porque a etapa 04 usa as mesmas regras.

In [ ]:
from src.consorcio import decompor_lance, complemento_cartao

lance = decompor_lance(p)
rotulos = {
    "carta": "Carta",
    "lance_total": "Lance total",
    "lance_embutido": "  parte embutida (sai da carta)",
    "lance_recurso_proprio": "  parte com recurso próprio",
    "credito_liquido": "Crédito líquido",
    "sobra_recurso_proprio": "Sobra de recurso próprio",
}
for chave, rotulo in rotulos.items():
    print(f"{rotulo:<32} R$ {lance[chave]:>10,.2f}")

In [ ]:
import pandas as pd

u = p["universo"]
tabela = pd.DataFrame([complemento_cartao(p, preco) for preco in (u["fipe_min"], u["fipe_max"])])
tabela.set_index("preco_carro").round(2)

## Decisões e observações

- **Validação automática.** `carregar_premissas()` agora confere o YAML toda vez que é chamada, em
  qualquer notebook. Os limites usados (ex.: taxa de desconto entre −5% e 20%) só pegam erro de
  digitação, como escrever `54` em vez de `0.54`; não julgam se a premissa é boa.
- **Fontes pendentes** ficam marcadas no próprio YAML com `FONTE PENDENTE`, e esta etapa lista todas.
  Assim a lista nunca fica desatualizada em relação ao arquivo.
- **Nova seção `dados`** no YAML: nome do dataset, série do IPCA e caminhos. Evita deixar esses
  valores fixos no código das próximas etapas.
- **Lance:** `pct_lance_total` (54%) inclui o embutido (25%); os 29% restantes saem do recurso próprio.
  O nome antigo `pct_lance_livre` foi trocado porque induzia a erro. A validação confere se o recurso
  próprio cobre essa parte do lance.
- **Carta:** R$ 85.758,91, confirmada pelo autor.
- **Carro mais barato que o disponível** (crédito líquido + sobra de recurso próprio): não há
  cartão, e a sobra abate o saldo devedor do consórcio (`abatimento_saldo_devedor`). O efeito nas
  parcelas restantes entra na etapa 04. A planilha devolvia −sobra no cartão nesse caso.